# 08 · Abstract Recovery & Topic-Model Refit

What this notebook covers: the NIH RePORTER + NSF Award Search abstract backfill
(`src/backfill_nih_reporter.py`, `src/backfill_nsf_awards.py`), how it was adopted into the
pipeline (`src/build_dataset.py`'s `_apply_abstract_backfill`, gap-fill only), and the BERTopic
refit that followed. This is a report on work already done, not a driver of it — it reads
already-computed artifacts (parquet/JSON) and needs only light dependencies (pandas/matplotlib/
seaborn), not `bertopic`/`torch`/`umap`.

**A note on "before" numbers below**: several are *historical, documented facts* from this
project's own record (`CLAUDE.md`, `docs/data_quality_report.md`, `docs/TOPIC_WORK_EXECUTION_REPORT.md`)
rather than recomputed live, since the pre-backfill pipeline state no longer exists on disk —
`grants.parquet` today already has the backfill applied. Each historical figure is called out
explicitly where used, so it's never ambiguous which numbers are "read from a file right now"
vs. "a citation of an earlier, already-verified measurement."

Do **not** commit notebook outputs (`nbstripout`, or strip manually before commit).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'data' / 'processed').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

PROC = REPO_ROOT / 'data' / 'processed'
OUTPUTS = REPO_ROOT / 'outputs'
BACKFILL = REPO_ROOT / 'data' / 'nih_nsf_backfill'
FIGDIR = OUTPUTS
FIGDIR.mkdir(exist_ok=True)

grants = pd.read_parquet(PROC / 'grants.parquet')
grants['grant_id'] = grants['grant_id'].astype(str)
faculty_grants = pd.read_parquet(PROC / 'faculty_grants.parquet')
topic_assignments = pd.read_parquet(PROC / 'topic_assignments.parquet')

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'grants.parquet: {grants.shape}')

## 1 · The coverage problem, before

**Historical figures** (`CLAUDE.md`, `docs/data_quality_report.md`), not recomputed here — the
pre-backfill state no longer exists in `grants.parquet`:

- **740 of 2,676 grants (28%)** had no abstract text at all going into this work.
- The **NIH post-2019 abstract "cliff"**: NIH/NIH-SubAward abstract coverage collapsed from 64%
  (2019) to essentially 0% from 2021 onward — a data-collection artifact in the internal upload
  system, not a real decline in NIH funding to NEU (NIH grant *counts* held steady the whole
  time; only the abstract text behind each one stopped arriving).
- This mattered for the topic model specifically because BERTopic's assignment confidence was
  already found to barely depend on having abstract text at all (nb07 / `TOPIC_ANALYSIS_COMPENDIUM.md`:
  ~28.0% unassigned for grants with an abstract vs. ~27.6% for title-only grants) — so backfilling
  text was motivated by wanting the *keyword lists that define each topic* to be evidence-backed,
  not by an expectation that it would reshape the clustering dramatically. Section 5 below checks
  whether that finding still holds now that titles were replaced by real text for many grants.

## 2 · The backfill

Recovery rates and composition, read live from `data/nih_nsf_backfill/*.parquet` — the actual
outputs of the two live API backfill runs.

In [ ]:
nih = pd.read_parquet(BACKFILL / 'backfill_nih_abstracts.parquet')
nsf = pd.read_parquet(BACKFILL / 'backfill_nsf_abstracts.parquet')

print(f"NIH backfill rows: {len(nih)}  ({(nih.abstract_source=='nih_reporter').sum()} nih_reporter, "
      f"{(nih.abstract_source=='nih_reporter_parent').sum()} nih_reporter_parent)")
print(f'NSF backfill rows: {len(nsf)}  (all nsf_api)')

# NIH-SubAward: the highest-risk, highest-yield slice (parent-center fallback risk, but also
# the only slice where every single grant was previously text-less).
sub_grants = grants[grants.agencyname == 'National Institutes of Health - SubAward']
print(f"\nNIH-SubAward: {sub_grants.grant_id.isin(nih.grant_id).sum()} / {len(sub_grants)} ""recovered")

In [ ]:
# Recovery rate against how many were text-less BEFORE this backfill. That "before" population
# can't be reconstructed exactly from today's grants.parquet (it already has the backfill
# applied) -- so these two headline rates are read from the backfill scripts' OWN reports, which
# measured them directly against the true pre-backfill state at the time each one ran.
for name, report in [('NIH', 'nih_reporter_backfill_report.md'), ('NSF', 'nsf_backfill_report.md')]:
    text = (BACKFILL / report).read_text()
    headline = next(l for l in text.splitlines() if l.startswith('Recovered abstracts for those grants'))
    going_in = next(l for l in text.splitlines() if 'text-less grants going in' in l)
    print(f'{name}: {going_in.strip()}')
    print(f'{name}: {headline.strip()}\n')

**The 5 `nih_reporter_parent` grants** are the one real risk this backfill carried: a
subaward whose specific subproject had no abstract of its own borrowed its **parent center's**
text instead (e.g. a P41/P42/P01 resource-center record). That's real, on-topic text — but not
*that grant's* text, so it's stored in `grants.parquet` for display but excluded from the
topic-model fit (`src.clean_text.LOW_TRUST_ABSTRACT_SOURCES`, applied in
`build_specter2_embeddings.py` and `topics_bertopic.py`). Verified below: those 5 grants are
embedded/clustered as title-only, exactly as if the text had never been recovered.

In [ ]:
manifest = pd.read_parquet(PROC / 'specter2_doc_manifest.parquet')
low_trust = manifest[manifest.abstract_source == 'nih_reporter_parent']
print(low_trust[['doc_id', 'title_chars', 'abstract_chars']].to_string(index=False))
assert (low_trust.abstract_chars == 0).all() and (low_trust.title_chars > 0).all()
print('\nconfirmed: all 5 embedded title-only (0 abstract_chars, title_chars > 0)')

**The awardee-organization audit** — RePORTER's own `organization.org_name` field on each
recovered record gives an independent, funder-sourced check on the pre-hire attribution caveat
(`CLAUDE.md` caveat #1: grants are attributed to a faculty member even when the award predates
their NEU hire). Cross-tabbed here against `faculty_grants.neu_status`, which is derived
independently from HR hire dates — two unrelated sources that should agree if both are honest
about the same underlying fact.

In [ ]:
nih_ab = nih[nih.abstract_source == 'nih_reporter']  # exclude the 5 parent-fallback rows
neu_status = faculty_grants[['grant_id', 'neu_status']].drop_duplicates('grant_id')
audit = nih_ab.merge(grants[['grant_id', 'agencyname']], on='grant_id', how='left') \
               .merge(neu_status, on='grant_id', how='left')

non_neu = audit[audit.awardee_org.str.upper().str.strip() != 'NORTHEASTERN UNIVERSITY']
non_neu_non_sub = non_neu[non_neu.agencyname != 'National Institutes of Health - SubAward']
print(f'Non-NEU-awardee rows (excl. SubAward, which is never NEU-awarded by design): {len(non_neu_non_sub)}')
print(non_neu_non_sub.neu_status.value_counts())
print()
print('Of NEU-awardee rows, for comparison:')
neu_ok = audit[(audit.awardee_org.str.upper().str.strip() == 'NORTHEASTERN UNIVERSITY')
               & (audit.agencyname != 'National Institutes of Health - SubAward')]
print(neu_ok.neu_status.value_counts())

Two independent signals substantially agree: most non-NEU-awardee grants are also flagged
`prior_institution` by the hire-date logic, and most NEU-awardee grants are `earned_at_neu`. They
don't agree perfectly (a fiscal-year-specific RePORTER record can reflect an institutional
transfer year that doesn't match our own start-date snapshot) — but the overall pattern
corroborates, rather than contradicts, the existing caveat.

## 3 · Adoption decisions

**Gap-fill only.** The backfill only writes text where `grants.parquet`'s `abstract` was
currently empty — it never overwrites existing `internal` text, even where the backfill scripts
found an "updated/longer" version. Before/after `abstract_source` coverage:

In [ ]:
counts = grants.abstract_source.value_counts()
counts.index = counts.index.map(lambda s: s if s else '(none)')
ax = counts.sort_values().plot.barh(figsize=(6, 3.5))
ax.set_xlabel('grants'); ax.set_title('grants.parquet abstract_source, after adoption')
for i, v in enumerate(counts.sort_values()):
    ax.text(v, i, f'  {v}', va='center')
plt.tight_layout()
plt.savefig(FIGDIR / 'w8_abstract_source_counts.png', dpi=150, bbox_inches='tight')
plt.show()
counts

**The AcAn export cross-check.** A refreshed internal-upload-system export
(`DataSet/AcAn Grants 2026-08-13.xlsx`) was estimated, *before* the live backfill existed, to
recover 198 grants (187 NIH + 11 NSF) — a historical figure from `docs/data_quality_report.md` §9.
Re-run *after* the live backfill (`scripts/_check_new_abstracts.py`), it recovers only:

In [ ]:
recoverable = pd.read_parquet(PROC / 'new_abstract_recovery.parquet')
print(f'AcAn net-new recoverable grants, post-backfill: {len(recoverable)}  (was 198 pre-backfill)')
recoverable

198 → 3: the live RePORTER/NSF backfill turned out to be a near-total superset of what the
AcAn refresh would have added. Given the tiny remaining yield, adopting the AcAn export itself
was dropped from this pass rather than building the extra plumbing (persisting its abstract
text, wiring a third backfill source) for 3 grants — documented in
`docs/data_quality_report.md` §9, not acted on further here.

## 4 · The refit itself

Corpus composition shift — the title-only fraction (data availability, not modeling exclusion;
see `titleOnly` vs `modelTitleOnly` in `src/build_viz_data.py`) dropped from 740/2,676 (28%,
historical) to:

In [ ]:
title_only_now = (grants.abstract.astype(str).str.strip() == '').sum()
print(f'{title_only_now} / {len(grants)}  ({100*title_only_now/len(grants):.1f}%)')

**The `min_cluster_size` sweep.** The corpus composition shift above was large enough that the
old default (`MIN_CLUSTER_SIZE=25`) was worth re-sweeping rather than assumed —
`python -m src.tune_bertopic`, 5 sizes × 3 seeds:

In [ ]:
sweep = json.loads((OUTPUTS / 'bertopic_sweep.json').read_text())
sweep_df = pd.DataFrame(sweep['runs'] if 'runs' in sweep else sweep)
sweep_df

**The old default of 25 turned out to be unstable on this corpus** — across 3 seeds it produced
anywhere from a degenerate 4-topic fit (one 2,503-doc mega-cluster, 91% of the corpus, in a
single `-1`/noise-free collapse) to 27 reasonable topics, a coin-flip on whether the fit
degenerates. `mcs=20` was the tightest value that stayed stable across all 3 seeds (32±0.5
topics, ~24.5% noise) and was chosen for the actual fit — see the comment above
`MIN_CLUSTER_SIZE` in `src/topics_bertopic.py`.

In [ ]:
diag = json.loads((OUTPUTS / 'bertopic_diagnostics.json').read_text())
print('New fit diagnostics:', diag)

# Historical reference (CLAUDE.md, docs/TOPIC_WORK_EXECUTION_REPORT.md) -- NOT recomputed,
# the old bertopic_model/ was overwritten by this refit.
OLD = {'n_topics': 25, 'unassigned_n': 808, 'unassigned_dollars_m': 607, 'unassigned_share': 0.278}

ARTIFACT_TOPIC_ID = 14  # see src/build_viz_aggregates.py
# topic_assignments.parquet also covers the 65 M2 orphan pseudo-docs ('orphan-<id>'), which
# aren't grants at all -- restrict to real grant_ids before comparing against len(grants), or
# the count and the % of it silently come from two different populations.
grants_ta = topic_assignments[topic_assignments.doc_id.isin(set(grants.grant_id))]
n_noise = int((grants_ta.topic_id == -1).sum())
n_artifact = int((grants_ta.topic_id == ARTIFACT_TOPIC_ID).sum())
total_dollars = grants.totaldollars.sum()
unassigned_dollars = grants[grants.grant_id.isin(
    grants_ta.loc[grants_ta.topic_id.isin([-1, ARTIFACT_TOPIC_ID]), 'doc_id']
)].totaldollars.sum()

print(f"\nOld: {OLD['n_topics']} topics, {OLD['unassigned_n']} unassigned "
      f"(${OLD['unassigned_dollars_m']}M, {100*OLD['unassigned_share']:.1f}%)")
print(f"New: {diag['n_topics']} topics, {n_noise + n_artifact} unassigned "
      f"(${unassigned_dollars/1e6:.0f}M, {100*(n_noise+n_artifact)/len(grants):.1f}% of grants, "
      f"{100*unassigned_dollars/total_dollars:.1f}% of dollars)")

**Did `ARTIFACT_TOPIC_ID`'s documented cause dissolve, as predicted?** Only partly. The old
artifact bucket (topic 11, pre-refit) was "28 of 62 docs are placeholder 'Grant' title-only
ONR/NIH-sub records." The NIH-SubAward backfill gave 100% of those grants real text — so if the
NIH-sub portion of that mix were the main driver, this bucket should have shrunk or vanished.

In [ ]:
artifact_docs = topic_assignments[topic_assignments.topic_id == ARTIFACT_TOPIC_ID]
artifact_grants = grants[grants.grant_id.isin(artifact_docs.doc_id)]
print(f'New artifact bucket (topic {ARTIFACT_TOPIC_ID}): {len(artifact_grants)} grants')
print(artifact_grants.agencyname.value_counts())
placeholder = (artifact_grants.grantname.astype(str).str.strip() == 'Grant').sum()
print(f"\nplaceholder-titled (exactly 'Grant'): {placeholder} / {len(artifact_grants)}")

**It persisted, and for exactly the reason the backfill couldn't reach**: the same **28**
placeholder-titled grants as before, now **100% Office of Naval Research** (ONR has no public
abstract API — explicitly out of scope for this backfill, see `CLAUDE.md`). The NIH-SubAward
docs that used to share this bucket are gone (they have real text now, and cluster elsewhere) —
so the bucket shrank from 62 to 51 docs and its *composition* got cleaner, but its *core cause*
is untouched, because that cause was never addressable by this particular backfill.

## 5 · Does "titles carry most of the signal" still hold?

The original finding (documented in `CLAUDE.md`, from nb07 / `TOPIC_ANALYSIS_COMPENDIUM.md`):
unassigned rate ~28.0% for grants with an abstract vs. ~27.6% for title-only grants — missing
the abstract barely hurt assignment confidence. Recomputed on the new fit:

In [ ]:
coverage = json.loads((REPO_ROOT / 'docs' / 'TopicVizPrototypes' / 'data' / 'coverage.json').read_text())
ct = coverage['crosstab']
abs_rate = ct['abs_unassigned'] / (ct['abs_assigned'] + ct['abs_unassigned'])
title_rate = ct['title_unassigned'] / (ct['title_assigned'] + ct['title_unassigned'])
print(f"New: unassigned rate {100*abs_rate:.1f}% (has abstract) vs {100*title_rate:.1f}% (title-only)")
print('Old (historical): 28.0% (has abstract) vs 27.6% (title-only)')

**This looks like a reversal — title-only grants now assign *more* confidently — but it's a
composition effect, not evidence titles got more informative.** The 740 title-only grants before
this work were a broad mix across agencies; the 286 left now are a much narrower, more
internally-consistent population:

In [ ]:
title_only = grants[grants.abstract.astype(str).str.strip() == '']
print(title_only.agencyname.value_counts().head(8))
top6_share = title_only.agencyname.value_counts().head(6).sum() / len(title_only)
print(f"\ntop 6 agencies (ONR/ARO/NASA/AFRO/DOE/NOAA) = {100*top6_share:.0f}% of all title-only grants")

ONR, ARO, NASA, AFRO, DOE, and NOAA together are ~83% of what's left title-only — almost
entirely defense/aerospace-adjacent funders whose grant titles tend to follow terse, consistent,
jargon-heavy conventions. That's a more clusterable population than the old, broader title-only
mix (which still included plenty of NIH/NSF grants that have since gained real text). The honest
read: **titles still carry real signal for BERTopic**, as originally found — but the specific
28%-vs-28% comparison doesn't carry over unchanged, because the population on one side of it
changed underneath it. Worth remembering the next time this crosstab gets re-run.

## 6 · What's still open

- **Investigator/co-PI proposals were never merged** into `faculty_grants.parquet` —
  `data/nih_nsf_backfill/investigator_faculty_proposals*.parquet` holds ~1,356 NSF and ~433 NIH
  proposed links (NIH ones should be filtered to `is_neu_org == True` first — a third of the raw
  NIH matches are the prime institution's own PI on a subaward, not an NEU person). Adopting them
  would reorder every funding-credit-model leaderboard, so it's deliberately deferred.
- **The AcAn export's 3 residual grants** are visible in `new_abstract_recovery.parquet` but not
  adopted (§3).
- **The PI's EnricoVis apps** (`docs/EnricoVis/{grant_atlas,topic_islands,topic_hierarchy}.html`)
  now render this new fit — their own HTML/JS is untouched, only the 3 JSON data files
  (`grants_umap.json`, `topics.json`, `grants_hier.json`) changed. Worth telling him the topic
  count moved from 25 to 32 before he notices on his own.
- **A real-browser pass on the three `TopicVizPrototypes` pages is still required** before
  treating this refit as publish-ready — no browser is available in this working environment, so
  nothing in this notebook (or the automated `_check_topicviz.py` checks) confirms actual
  rendering, only that the data is structurally well-formed.